# CSL7110 – Assignment 4: Clustering & PageRank

**Parts covered:**
1. Part 1 – K-Center (Farthest-First) & K-Means++ Clustering on UCI Spam dataset  
2. Part 2 – Inverted Index Web Search Engine  
3. Part 3 – PageRank on Apache Spark  

---

## Environment Setup
Install / import all required libraries.

In [35]:
import math
import time
import random
import string
import os
import re
from collections import defaultdict

from pyspark import SparkContext, SparkConf
from pyspark.mllib.linalg import Vectors

print("All imports successful.")

All imports successful.


---
# PART 1 – Clustering
### Algorithms: Farthest-First Traversal (K-Center) and K-Means++

## 1.1 – Initialize Spark

In [36]:
conf = SparkConf().setAppName("Clustering").setMaster("local[*]")

try:
    sc.stop()
except Exception:
    pass

sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")
print("Spark context started:", sc.version)

Spark context started: 3.5.1


## 1.2 – `readVectorsSeq` : Load the UCI Spam dataset

In [37]:
def readVectorsSeq(filename):
    points = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            points.append(Vectors.dense([float(x) for x in line.split(',')]))
    return points


DATA_PATH = "data/Q1- UCI Spam clustering/spambase.data"
P = readVectorsSeq(DATA_PATH)

print(f"Loaded {len(P)} points, each with {len(P[0])} dimensions.")

Loaded 4601 points, each with 58 dimensions.


## 1.3 – `kcenter` : Farthest-First Traversal

**Algorithm:**
1. Pick the first center randomly.
2. At each subsequent step, pick the point that is **farthest** from the current set of centers.
3. Repeat until `k` centers are chosen.

Time complexity: **O(|P| × k)**

In [38]:
def kcenter(P, k):
    if k >= len(P):
        return list(P)

    centers = [random.choice(P)]
    min_dist = [Vectors.squared_distance(p, centers[0]) for p in P]

    for _ in range(k - 1):
        farthest_idx = max(range(len(P)), key=lambda i: min_dist[i])
        new_center = P[farthest_idx]
        centers.append(new_center)

        for i, p in enumerate(P):
            d = Vectors.squared_distance(p, new_center)
            if d < min_dist[i]:
                min_dist[i] = d

    return centers


print("kcenter function defined.")

kcenter function defined.


## 1.4 – `kmeansPP` : K-Means++ Seeding

**Algorithm (D² sampling):**
1. Pick the first center uniformly at random.
2. At each step, pick the next center with probability proportional to its **squared distance** from the nearest existing center.
3. Repeat until `k` centers are chosen.

Time complexity: **O(|P| × k)**

In [39]:
def kmeansPP(P, k):
    if k >= len(P):
        return list(P)

    centers = [random.choice(P)]
    min_dist = [Vectors.squared_distance(p, centers[0]) for p in P]

    for _ in range(k - 1):
        total = sum(min_dist)
        r = random.uniform(0, total)
        cumulative = 0.0
        chosen_idx = 0

        for i, d in enumerate(min_dist):
            cumulative += d
            if cumulative >= r:
                chosen_idx = i
                break

        new_center = P[chosen_idx]
        centers.append(new_center)

        for i, p in enumerate(P):
            d = Vectors.squared_distance(p, new_center)
            if d < min_dist[i]:
                min_dist[i] = d

    return centers


print("kmeansPP function defined.")

kmeansPP function defined.


## 1.5 – `kmeansObj` : K-Means Objective Function

In [40]:
def kmeansObj(P, C):
    total_dist = 0.0
    for p in P:
        total_dist += min(Vectors.squared_distance(p, c) for c in C)
    return total_dist / len(P)


print("kmeansObj function defined.")

kmeansObj function defined.


## 1.6 – Main Program: Run all three experiments

Adjust `k` and `k1` as needed (k < k1).

In [41]:
k = 10
k1 = 50

random.seed(42)

print("=" * 60)
print(f"Experiment 1: kcenter(P, k={k})")
print("=" * 60)

start = time.time()
C_kcenter = kcenter(P, k)
elapsed = time.time() - start

print(f"Running time of kcenter: {elapsed:.4f} seconds")
print(f"Number of centers returned: {len(C_kcenter)}")

print()
print("=" * 60)
print(f"Experiment 2: kmeansPP(P, k={k}) -> kmeansObj")
print("=" * 60)

C_kmeanspp = kmeansPP(P, k)
obj2 = kmeansObj(P, C_kmeanspp)
print(f"K-Means Objective (kmeansPP, k={k}): {obj2:.6f}")

print()
print("=" * 60)
print(f"Experiment 3: kcenter(P, k1={k1}) -> kmeansPP(X, k={k}) -> kmeansObj")
print("=" * 60)

X = kcenter(P, k1)
C_combined = kmeansPP(X, k)
obj3 = kmeansObj(P, C_combined)
print(f"K-Means Objective (coreset approach, k={k}, k1={k1}): {obj3:.6f}")

print()
print("=" * 60)
print("Summary")
print("=" * 60)
print(f"  Experiment 2 objective (kmeans++ direct):  {obj2:.6f}")
print(f"  Experiment 3 objective (kcenter coreset):  {obj3:.6f}")
print()
print("Lower objective is better.")

Experiment 1: kcenter(P, k=10)
Running time of kcenter: 0.1828 seconds
Number of centers returned: 10

Experiment 2: kmeansPP(P, k=10) -> kmeansObj
K-Means Objective (kmeansPP, k=10): 25429.791217

Experiment 3: kcenter(P, k1=50) -> kmeansPP(X, k=10) -> kmeansObj
K-Means Objective (coreset approach, k=10, k1=50): 78592.549103

Summary
  Experiment 2 objective (kmeans++ direct):  25429.791217
  Experiment 3 objective (kcenter coreset):  78592.549103

Lower objective is better.


---
# PART 2 – Web Search with Inverted Index

We implement the full class hierarchy:
`MySet → Position → WordEntry → PageIndex → PageEntry → MyHashTable → InvertedPageIndex → SearchEngine`

## 2.1 – Stop Words, Punctuation & Stemming Rules

In [42]:
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is', 'are',
    'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

PUNCTUATION = set('{}[]<>=(). ,;\'"?#!-:')

PLURAL_MAP = {
    'stacks': 'stack',
    'structures': 'structure',
    'applications': 'application',
}


def normalize_word(word):
    cleaned = ''.join(' ' if ch in PUNCTUATION else ch for ch in word.lower()).strip()
    if not cleaned or cleaned in STOP_WORDS:
        return None
    return PLURAL_MAP.get(cleaned, cleaned)


def tokenize(text):
    cleaned_text = ''.join(' ' if ch in PUNCTUATION else ch for ch in text)
    raw_tokens = cleaned_text.split()

    tokens = []
    for pos, raw in enumerate(raw_tokens, start=1):
        norm = normalize_word(raw)
        if norm is not None:
            tokens.append((norm, pos))
    return tokens


print("Tokenization utilities defined.")

sample = "Data structures is the study of structures for storing data."
print("Sample tokenization:", tokenize(sample))

Tokenization utilities defined.
Sample tokenization: [('data', 1), ('structure', 2), ('study', 5), ('structure', 7), ('storing', 9), ('data', 10)]


## 2.2 – `MySet` : Custom Set with Union & Intersection

In [43]:
class MySet:
    def __init__(self):
        self._data = set()

    def addElement(self, element):
        self._data.add(element)

    def union(self, otherSet):
        result = MySet()
        result._data = self._data | otherSet._data
        return result

    def intersection(self, otherSet):
        result = MySet()
        result._data = self._data & otherSet._data
        return result

    def __contains__(self, item):
        return item in self._data

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"MySet({self._data})"


print("MySet defined.")

MySet defined.


## 2.3 – `Position` : (PageEntry, wordIndex) Tuple

In [44]:
class Position:
    def __init__(self, pageEntry, wordIndex: int):
        self._pageEntry = pageEntry
        self._wordIndex = wordIndex

    def getPageEntry(self):
        return self._pageEntry

    def getWordIndex(self):
        return self._wordIndex

    def __repr__(self):
        return f"Position(page={self._pageEntry.getPageName()}, idx={self._wordIndex})"


print("Position defined.")

Position defined.


## 2.4 – `WordEntry` : All positions for a single word

In [45]:
class WordEntry:
    def __init__(self, word: str):
        self._word = word
        self._positions = []

    def addPosition(self, position):
        self._positions.append(position)

    def addPositions(self, positions: list):
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return list(self._positions)

    def getTermFrequency(self, pageName: str):
        count = sum(
            1 for pos in self._positions
            if pos.getPageEntry().getPageName() == pageName
        )
        if count == 0:
            return 0.0

        total = 0
        for pos in self._positions:
            if pos.getPageEntry().getPageName() == pageName:
                total = pos.getPageEntry().getTotalWords()
                break

        return count / total if total > 0 else 0.0

    def getWord(self):
        return self._word

    def __repr__(self):
        return f"WordEntry('{self._word}', positions={len(self._positions)})"


print("WordEntry defined.")

WordEntry defined.


## 2.5 – `PageIndex` : Per-page word → positions mapping

In [46]:
class PageIndex:
    def __init__(self):
        self._index = {}

    def addPositionForWord(self, word: str, position):
        if word not in self._index:
            self._index[word] = WordEntry(word)
        self._index[word].addPosition(position)

    def getWordEntries(self):
        return list(self._index.values())

    def getWordEntry(self, word: str):
        return self._index.get(word, None)

    def containsWord(self, word: str):
        return word in self._index

    def __repr__(self):
        return f"PageIndex(words={list(self._index.keys())})"


print("PageIndex defined.")

PageIndex defined.


## 2.6 – `PageEntry` : Reads a webpage file and builds its PageIndex

In [47]:
WEBPAGES_DIR = "data/Q2- webSearch/webpages/"

class PageEntry:
    def __init__(self, pageName: str):
        self._pageName = pageName
        self._pageIndex = PageIndex()
        self._totalWords = 0

        filepath = os.path.join(WEBPAGES_DIR, pageName)
        self._load(filepath)

    def _load(self, filepath):
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()

        cleaned_text = ''.join(' ' if ch in PUNCTUATION else ch for ch in text)
        all_tokens = cleaned_text.split()
        self._totalWords = len(all_tokens)

        for word, pos in tokenize(text):
            self._pageIndex.addPositionForWord(word, Position(self, pos))

    def getPageName(self):
        return self._pageName

    def getPageIndex(self):
        return self._pageIndex

    def getTotalWords(self):
        return self._totalWords

    def __repr__(self):
        return f"PageEntry('{self._pageName}')"


print("PageEntry defined.")

PageEntry defined.


## 2.7 – `MyHashTable` : Word → WordEntry mapping

In [48]:
class MyHashTable:
    def __init__(self, size: int = 1024):
        self._size = size
        self._table = {}

    def getHashIndex(self, word: str) -> int:
        return hash(word) % self._size

    def addPositionsForWord(self, wordEntry):
        word = wordEntry.getWord()
        if word in self._table:
            self._table[word].addPositions(wordEntry.getAllPositionsForThisWord())
        else:
            new_entry = WordEntry(word)
            new_entry.addPositions(wordEntry.getAllPositionsForThisWord())
            self._table[word] = new_entry

    def getWordEntry(self, word: str):
        return self._table.get(word, None)

    def getAllWordEntries(self):
        return list(self._table.values())

    def __contains__(self, word):
        return word in self._table

    def __repr__(self):
        return f"MyHashTable(words={len(self._table)})"


print("MyHashTable defined.")

MyHashTable defined.


## 2.8 – `InvertedPageIndex` : Global index across all pages

In [49]:
class InvertedPageIndex:
    def __init__(self):
        self._hashTable = MyHashTable()
        self._pages = {}

    def addPage(self, pageEntry):
        name = pageEntry.getPageName()
        self._pages[name] = pageEntry

        for wordEntry in pageEntry.getPageIndex().getWordEntries():
            self._hashTable.addPositionsForWord(wordEntry)

    def getPagesWhichContainWord(self, word: str):
        result = MySet()
        entry = self._hashTable.getWordEntry(word)
        if entry is None:
            return result

        seen = set()
        for pos in entry.getAllPositionsForThisWord():
            pe = pos.getPageEntry()
            name = pe.getPageName()
            if name not in seen:
                result.addElement(pe)
                seen.add(name)
        return result

    def getWordEntry(self, word: str):
        return self._hashTable.getWordEntry(word)

    def getAllPages(self):
        return list(self._pages.values())

    def getPageEntry(self, pageName: str):
        return self._pages.get(pageName, None)

    def totalPages(self):
        return len(self._pages)

    def __repr__(self):
        return f"InvertedPageIndex(pages={len(self._pages)}, words={len(self._hashTable._table)})"


print("InvertedPageIndex defined.")

InvertedPageIndex defined.


In [ ]:
for word_entry in sorted(engine._index._hashTable.getAllWordEntries(), key=lambda w: w.getWord()):
    positions = word_entry.getAllPositionsForThisWord()
    formatted = []

    for pos in positions:
        page_name = pos.getPageEntry().getPageName()
        word_index = pos.getWordIndex()
        formatted.append(f"({page_name}, {word_index})")

    print(f"{word_entry.getWord()} : {{{', '.join(formatted)}}}")

1946 : {(stack_datastructure_wiki, 299)}
7 : {(stack_oracle, 3)}
about : {(stack_cprogramming, 66), (stack_cprogramming, 217), (stack_cprogramming, 278)}
abstract : {(stack_datastructure_wiki, 2), (stack_datastructure_wiki, 23)}
abstractly : {(stack_datastructure_wiki, 106)}
accept : {(stack_datastructure_wiki, 163)}
access : {(stack_datastructure_wiki, 137)}
added : {(stack_datastructure_wiki, 54), (stack_cprogramming, 85), (stack_cprogramming, 131), (stack_cprogramming, 178)}
adding : {(stack_cprogramming, 118)}
addition : {(stack_datastructure_wiki, 277)}
additionally : {(stack_datastructure_wiki, 131)}
adds : {(stack_datastructure_wiki, 39)}
ai : {(stack_cprogramming, 304)}
alan : {(stack_datastructure_wiki, 305)}
alex : {(stack_cprogramming, 10)}
allain : {(stack_cprogramming, 11)}
allow : {(stack_oracle, 38)}
allows : {(stack_cprogramming, 117)}
already : {(stack_datastructure_wiki, 354), (stack_cprogramming, 206)}
also : {(stack_datastructure_wiki, 253), (stack_datastructure_wik

## 2.9 – `SearchEngine` : Action dispatcher for queries

The `SearchEngine` class orchestrates all operations:
- `addPage(pageName)`: Load and index a webpage
- `queryFindPagesWhichContainWord(word)`: Return pages containing the word
- `queryFindPositionsOfWordInAPage(word, pageName)`: Return word positions in a specific page


In [50]:
class SearchEngine:
    def __init__(self):
        self._index = InvertedPageIndex()

    def _addPage(self, pageName: str):
        try:
            self._index.addPage(PageEntry(pageName))
        except FileNotFoundError:
            print(f"[ERROR] File not found: {pageName}")

    def _queryFindPagesWhichContainWord(self, word: str):
        norm = normalize_word(word)
        if norm is None:
            print(f"No webpage contains word {word}")
            return

        pages = self._index.getPagesWhichContainWord(norm)
        if len(pages) == 0:
            print(f"No webpage contains word {word}")
            return

        names = sorted(pe.getPageName() for pe in pages)
        print(", ".join(names))

    def _queryFindPositionsOfWordInAPage(self, word: str, pageName: str):
        pageEntry = self._index.getPageEntry(pageName)
        if pageEntry is None:
            print(f"No webpage {pageName} found")
            return

        norm = normalize_word(word)
        if norm is None or not pageEntry.getPageIndex().containsWord(norm):
            print(f"Webpage {pageName} does not contain word {word}")
            return

        local_entry = pageEntry.getPageIndex().getWordEntry(norm)
        positions = sorted(
            pos.getWordIndex() for pos in local_entry.getAllPositionsForThisWord()
        )
        print(", ".join(str(p) for p in positions))

    def performAction(self, actionMessage: str):
        parts = actionMessage.strip().split()
        if not parts:
            return

        action = parts[0]

        if action == 'addPage':
            self._addPage(parts[1])
        elif action == 'queryFindPagesWhichContainWord':
            self._queryFindPagesWhichContainWord(parts[1])
        elif action == 'queryFindPositionsOfWordInAPage':
            self._queryFindPositionsOfWordInAPage(parts[1], parts[2])
        else:
            print(f"[WARN] Unknown action: {action}")


print("SearchEngine defined.")

SearchEngine defined.


In [61]:
# Part 2 demo: show the methods working one by one with real data
demo_page = PageEntry("stack_datastructure_wiki")
demo_page_2 = PageEntry("stackoverflow")

demo_word = demo_page.getPageIndex().getWordEntries()[0].getWord()
demo_word_entry = demo_page.getPageIndex().getWordEntry(demo_word)
demo_positions = demo_word_entry.getAllPositionsForThisWord()

print("1) Position")
demo_pos = demo_positions[0]
print("   page:", demo_pos.getPageEntry().getPageName())
print("   index:", demo_pos.getWordIndex())
print()

print("2) WordEntry.addPosition / addPositions / getAllPositionsForThisWord / getTermFrequency")
test_entry = WordEntry(demo_word)
test_entry.addPosition(demo_positions[0])
if len(demo_positions) > 1:
    test_entry.addPositions(demo_positions[1:3])
else:
    test_entry.addPositions(demo_positions)
print("   word:", test_entry.getWord())
print("   positions:", [(p.getPageEntry().getPageName(), p.getWordIndex()) for p in test_entry.getAllPositionsForThisWord()])
print("   term frequency:", test_entry.getTermFrequency(demo_page.getPageName()))
print()

print("3) PageIndex methods")
test_index = PageIndex()
test_index.addPositionForWord(demo_word, demo_positions[0])
test_index.addPositionForWord(demo_word, demo_positions[0])
other_word = demo_page.getPageIndex().getWordEntries()[1].getWord()
other_positions = demo_page.getPageIndex().getWordEntry(other_word).getAllPositionsForThisWord()
test_index.addPositionForWord(other_word, other_positions[0])
print("   contains demo_word:", test_index.containsWord(demo_word))
print("   getWordEntry(demo_word):", test_index.getWordEntry(demo_word))
print("   word entries:", [we.getWord() for we in test_index.getWordEntries()])
print()

print("4) MySet methods")
set_a = MySet()
set_b = MySet()
set_a.addElement("alpha")
set_a.addElement("beta")
set_b.addElement("beta")
set_b.addElement("gamma")
print("   set_a:", list(set_a))
print("   set_b:", list(set_b))
print("   union:", list(set_a.union(set_b)))
print("   intersection:", list(set_a.intersection(set_b)))
print()

print("5) MyHashTable methods")
test_table = MyHashTable()
test_table.addPositionsForWord(demo_word_entry)
print("   hash index:", test_table.getHashIndex(demo_word))
print("   getWordEntry:", test_table.getWordEntry(demo_word))
print("   all entries:", [we.getWord() for we in test_table.getAllWordEntries()])
print()

print("6) InvertedPageIndex methods")
inv_index = InvertedPageIndex()
inv_index.addPage(demo_page)
inv_index.addPage(demo_page_2)
print("   total pages:", inv_index.totalPages())
print("   all pages:", [p.getPageName() for p in inv_index.getAllPages()])
print("   getPageEntry:", inv_index.getPageEntry(demo_page.getPageName()))
print("   pages with word:", [p.getPageName() for p in inv_index.getPagesWhichContainWord(demo_word)])
print("   global word entry:", inv_index.getWordEntry(demo_word))
print()

print("7) SearchEngine methods")
demo_engine = SearchEngine()
demo_engine.performAction(f"addPage {demo_page.getPageName()}")
demo_engine.performAction(f"addPage {demo_page_2.getPageName()}")
demo_engine.performAction(f"queryFindPagesWhichContainWord {demo_word}")
demo_engine.performAction(f"queryFindPositionsOfWordInAPage {demo_word} {demo_page.getPageName()}")

1) Position
   page: stack_datastructure_wiki
   index: 1

2) WordEntry.addPosition / addPositions / getAllPositionsForThisWord / getTermFrequency
   word: stack
   positions: [('stack_datastructure_wiki', 1), ('stack_datastructure_wiki', 14), ('stack_datastructure_wiki', 71)]
   term frequency: 0.006507592190889371

3) PageIndex methods
   contains demo_word: True
   getWordEntry(demo_word): WordEntry('stack', positions=2)
   word entries: ['stack', 'abstract']

4) MySet methods
   set_a: ['alpha', 'beta']
   set_b: ['beta', 'gamma']
   union: ['alpha', 'beta', 'gamma']
   intersection: ['beta']

5) MyHashTable methods
   hash index: 759
   getWordEntry: WordEntry('stack', positions=26)
   all entries: ['stack']

6) InvertedPageIndex methods
   total pages: 2
   all pages: ['stack_datastructure_wiki', 'stackoverflow']
   getPageEntry: PageEntry('stack_datastructure_wiki')
   pages with word: ['stack_datastructure_wiki', 'stackoverflow']
   global word entry: WordEntry('stack', positio

## 2.10 – Run all actions from `actions.txt` and compare with `answers.txt`

In [59]:
ACTIONS_FILE = "data/Q2- webSearch/actions.txt"
ANSWERS_FILE = "data/Q2- webSearch/answers.txt"

with open(ACTIONS_FILE, 'r') as f:
    actions = [line.strip() for line in f if line.strip()]

with open(ANSWERS_FILE, 'r') as f:
    answers = [line.strip() for line in f if line.strip()]

print(f"Total actions : {len(actions)}")
print(f"Total answers : {len(answers)}")
print()

import io
import sys
from contextlib import redirect_stdout

engine = SearchEngine()
outputs = []

for action in actions:
    if action.startswith('addPage'):
        engine.performAction(action)
    else:
        captured = io.StringIO()
        with redirect_stdout(captured):
            engine.performAction(action)
        outputs.append((action, captured.getvalue().strip()))

print(f"{'Action':<60} {'Got':<40} {'Expected':<40} {'Match'}")
print("-" * 160)
for i, (action, got) in enumerate(outputs):
    expected = answers[i] if i < len(answers) else "N/A"
    match = "✓" if got == expected else "✗"
    print(f"{action:<60} {got:<40} {expected:<40} {match}")

Total actions : 17
Total answers : 11

Action                                                       Got                                      Expected                                 Match
----------------------------------------------------------------------------------------------------------------------------------------------------------------
queryFindPagesWhichContainWord delhi                         No webpage contains word delhi           No webpage contains word delhi           ✓
queryFindPagesWhichContainWord stack                         stack_datastructure_wiki                 stack_datastructure_wiki                 ✓
queryFindPagesWhichContainWord wikipedia                     stack_datastructure_wiki                 stack_datastructure_wiki                 ✓
queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki Webpage stack_datastructure_wiki does not contain word magazines Webpage stack_datastructure_wiki does not contain word magazines ✓
queryFindPagesWhi

---
# PART 3 – PageRank on Apache Spark

Dataset: `data/pagerank/small.txt` and `data/pagerank/whole.txt`  
(Download from https://github.com/pnijhara/PySpark-PageRank/tree/main/graph)

**PageRank update rule:**
$$r^{(i)} = \frac{1-\beta}{n}\mathbf{1} + \beta M r^{(i-1)}$$

β = 0.8, iterations = 40.

## 3.1 – Restart / reuse Spark Context

In [52]:
import os
import sys
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

conda_site_packages = os.path.join(
    sys.prefix,
    'lib',
    f'python{sys.version_info.major}.{sys.version_info.minor}',
    'site-packages'
 )

conf = (
    SparkConf()
    .setAppName("PageRank")
    .setMaster("local[*]")
    .set("spark.pyspark.python", sys.executable)
    .set("spark.pyspark.driver.python", sys.executable)
    .set("spark.executorEnv.PYTHONPATH", conda_site_packages)
    .set("spark.sql.execution.arrow.pyspark.enabled", "false")
)

try:
    sc.stop()
except Exception:
    pass

sc = SparkContext(conf=conf)
spark = SparkSession.builder.config(conf=conf).getOrCreate()

print("Spark session is ready.")

Spark session is ready.


## 3.2 – Helper functions: load graph, build transition matrix M

In [53]:
def load_graph(filepath):
    """Load graph edges from a text file and return unique edges plus node info."""
    raw = sc.textFile(filepath)

    edges_rdd = (
        raw
        .map(lambda line: line.strip().split())
        .filter(lambda parts: len(parts) >= 2)
        .map(lambda parts: (int(parts[0]), int(parts[1])))
        .distinct()
    )

    srcs = edges_rdd.map(lambda e: e[0])
    dsts = edges_rdd.map(lambda e: e[1])
    nodes = sorted(srcs.union(dsts).distinct().collect())
    n = len(nodes)

    return edges_rdd, n, nodes


def build_transition_rdd(edges_rdd):
    """Return transition entries as (dst, (src, 1/out_degree(src)))."""
    out_degree = edges_rdd.map(lambda e: (e[0], 1)).reduceByKey(lambda a, b: a + b)

    transition_rdd = (
        edges_rdd
        .join(out_degree)
        .map(lambda x: (x[1][0], (x[0], 1.0 / x[1][1])))
    )

    return transition_rdd


print("PageRank helpers loaded.")

PageRank helpers loaded.


## 3.3 – Iterative PageRank function

In [54]:
def run_pagerank(edges_rdd, n, beta=0.8, iterations=40):
    """Run iterative PageRank and return a dict: node -> score."""
    teleport = (1.0 - beta) / (n + 1e-9)
    transition_rdd = build_transition_rdd(edges_rdd).cache()

    srcs = edges_rdd.map(lambda e: e[0])
    dsts = edges_rdd.map(lambda e: e[1])
    all_nodes = srcs.union(dsts).distinct()

    r_rdd = all_nodes.map(lambda node: (node, 1.0 / n)).cache()

    for it in range(iterations):
        r_dict = dict(r_rdd.collect())
        r_broadcast = sc.broadcast(r_dict)

        contributions = (
            transition_rdd
            .map(lambda x: (x[0], x[1][1] * r_broadcast.value.get(x[1][0], 0.0)))
            .reduceByKey(lambda a, b: a + b)
        )

        r_new = contributions.map(lambda x: (x[0], teleport + beta * x[1]))

        existing_dsts = set(contributions.map(lambda x: x[0]).collect())
        no_inbound = all_nodes.filter(lambda node: node not in existing_dsts)
        fallback = no_inbound.map(lambda node: (node, teleport))

        r_rdd = r_new.union(fallback).cache()
        r_broadcast.unpersist()

        if (it + 1) % 10 == 0:
            print(f"  Iteration {it + 1}/{iterations} complete.")

    return dict(r_rdd.collect())


print("PageRank runner loaded.")

PageRank runner loaded.


## 3.4 – Validate on `small.txt` (expected top score ≈ 0.036)

In [30]:
import pyspark
import cloudpickle
print("PySpark version:", pyspark.__version__)
print("cloudpickle version:", cloudpickle.__version__)

PySpark version: 3.5.0
cloudpickle version: 2.2.1


In [55]:
SMALL_GRAPH = "data/pagerank/small.txt"

def ensure_active_spark_context():
    global sc
    try:
        _ = sc.defaultParallelism
    except Exception:
        conf_local = (
            SparkConf()
            .setAppName("PageRank")
            .setMaster("local[*]")
            .set("spark.pyspark.python", sys.executable)
            .set("spark.pyspark.driver.python", sys.executable)
        )
        sc = SparkContext.getOrCreate(conf=conf_local)
        sc.setLogLevel("ERROR")
        print("Spark context was inactive. Restarted it.")

ensure_active_spark_context()

print("Loading small graph...")
edges_small, n_small, nodes_small = load_graph(SMALL_GRAPH)
print(f"Nodes: {n_small}, Edges (unique): {edges_small.count()}")

print("\nRunning PageRank on small graph (beta=0.8, 40 iterations)...")
pr_small = run_pagerank(edges_small, n_small, beta=0.8, iterations=40)
sorted_small = sorted(pr_small.items(), key=lambda x: -x[1])

print("\nTop-5 nodes (small graph):")
for node, score in sorted_small[:5]:
    print(f"  Node {node:>6}  ->  {score:.6f}")

print(f"\nTop score: {sorted_small[0][1]:.6f}  (expected approx 0.036)")

Loading small graph...


Nodes: 100, Edges (unique): 950

Running PageRank on small graph (beta=0.8, 40 iterations)...
  Iteration 10/40 complete.
  Iteration 20/40 complete.
  Iteration 30/40 complete.
  Iteration 40/40 complete.

Top-5 nodes (small graph):
  Node     53  ->  0.035731
  Node     14  ->  0.034171
  Node     40  ->  0.033630
  Node      1  ->  0.030006
  Node     27  ->  0.029720

Top score: 0.035731  (expected approx 0.036)


## 3.5 – Run on full `whole.txt` graph (n=1000, m≈8192)

In [32]:
WHOLE_GRAPH = "data/pagerank/whole.txt"

try:
    ensure_active_spark_context()
except NameError:
    if 'sc' not in globals() or sc is None or sc._jsc is None:
        conf_local = (
            SparkConf()
            .setAppName("PageRank")
            .setMaster("local[*]")
            .set("spark.pyspark.python", sys.executable)
            .set("spark.pyspark.driver.python", sys.executable)
        )
        sc = SparkContext.getOrCreate(conf=conf_local)
        sc.setLogLevel("ERROR")
        print("Spark context was inactive. Restarted it.")

print("Loading whole graph...")
edges_whole, n_whole, nodes_whole = load_graph(WHOLE_GRAPH)
print(f"Nodes: {n_whole}, Edges (unique): {edges_whole.count()}")

print("\nRunning PageRank on whole graph (beta=0.8, 40 iterations)...")
pr_whole = run_pagerank(edges_whole, n_whole, beta=0.8, iterations=40)
sorted_whole = sorted(pr_whole.items(), key=lambda x: -x[1])

print("\n" + "=" * 50)
print("Results - whole graph")
print("=" * 50)

print("\nTop 5 nodes (highest PageRank):")
for node, score in sorted_whole[:5]:
    print(f"  Node {node:>6}  ->  {score:.8f}")

print("\nBottom 5 nodes (lowest PageRank):")
for node, score in sorted_whole[-5:]:
    print(f"  Node {node:>6}  ->  {score:.8f}")

Loading whole graph...
Nodes: 1000, Edges (unique): 8161

Running PageRank on whole graph (beta=0.8, 40 iterations)...
  Iteration 10/40 complete.
  Iteration 20/40 complete.
  Iteration 30/40 complete.
  Iteration 40/40 complete.

Results - whole graph

Top 5 nodes (highest PageRank):
  Node    263  ->  0.00202029
  Node    537  ->  0.00194334
  Node    965  ->  0.00192545
  Node    243  ->  0.00185263
  Node    285  ->  0.00182737

Bottom 5 nodes (lowest PageRank):
  Node    408  ->  0.00038780
  Node    424  ->  0.00035482
  Node     62  ->  0.00035315
  Node     93  ->  0.00035136
  Node    558  ->  0.00032860


## 3.6 – Stop Spark

In [33]:
sc.stop()
print("Spark context stopped.")

Spark context stopped.


---
## Summary

| Part | Task | Status |
|------|------|--------|
| 1 | `readVectorsSeq`, `kcenter`, `kmeansPP`, `kmeansObj` implemented | ✓ |
| 1 | Three experiments run with timing and objective values | ✓ |
| 2 | Full class hierarchy: `MySet → SearchEngine` | ✓ |
| 2 | All actions from `actions.txt` processed and verified vs `answers.txt` | ✓ |
| 3 | Iterative PageRank on Spark, validated on `small.txt` | ✓ |
| 3 | Top-5 / Bottom-5 nodes reported for `whole.txt` | ✓ |

In [34]:
print(f"Top score check: {sorted_small[0][1]:.6f}")
print(f"Rank sum check: {sum(pr_small.values()):.6f}")

Top score check: 0.035731
Rank sum check: 1.000000
